## Análisis del GRD para la desigualdad de acceso a hospitalización compleja

Utilizando la base de datos del GRD cruzada con: Base de Establecimientos (DIES), Camas Hospitalarias (DIES), Proyección de población del INE, CIE10 (diagnósticos) y estimación de la pobreza multidimensional (BIDAT), se realizará un análisis exploratorio de las variables clave:

In [1]:
import pandas as pd
from charset_normalizer import from_bytes
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import glob as gb

In [2]:
grd = pd.read_parquet('grd_procesado.parquet')

Primero, se entiende la estructura de nuestra base, conociendo sus variables, los tipos de datos que contiene, la cantidad de filas y columnas, etc.

In [ ]:
# Verificamos si hay valores nulos o faltantes
print("Valores nulos por columna:")
print(grd.isnull().sum())

# Verificamos valores únicos en algunas columnas clave
print("\nValores únicos en columna SEXO:")
print(grd['SEXO'].unique())

# print("\nValores únicos en columna TIPOALTA:")
# print(grd['TIPOALTA'].unique())

print("\nValores unicos de severidad")
print(grd['IR_29301_SEVERIDAD'].unique())

print("--- df.head() ---")
display(grd.head())          # primeras 5 filas

print("\n--- df.shape ---")
print(grd.shape)            

print("\n--- df.info() ---")
grd.info()                   # tipos + nulos

print("\n--- df.describe() ---")
display(grd.describe())                    # estadísticas numéricas

print("\n--- df.describe(include='all') ---")
display(grd.describe(include="all"))       # incluye categóricas

Se realizan diferentes filtros para poder conocer mejor a los pacientes a nivel demográfico, utilizando sexo y edad.

In [ ]:
plt.figure(figsize=(11, 6))
ax = sns.countplot(data=grd, x='SEXO')
plt.title('Distribución de pacientes por sexo')
plt.ylabel('Cantidad de pacientes')
plt.xticks(rotation=0)

# Añadir porcentaje en las barras
total = len(grd)
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.1f}%'
    x = p.get_x() + p.get_width() / 2
    y = p.get_height() /2
    ax.annotate(percentage, (x, y), ha='center', va='bottom', color='yellow', weight='bold')

plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
# Filter data and convert EDAD to float to ensure compatibility with KDE
# edad_data = grdtest.dropna(subset=['EDAD'])['EDAD'].astype(float)
sns.histplot(grd['EDAD'], bins=20, kde=True)
plt.title('Distribución de edades de los pacientes')
plt.xlabel('Edad (años)')
plt.ylabel('Cantidad de pacientes')
plt.show()

Se estudian variables clave, como la severidad de los pacientes y dias de estancia, para poder entender como se distribuyen en paciente de hospitalización compleja.

In [ ]:
grd['IR_29301_SEVERIDAD'] = grd.IR_29301_SEVERIDAD.astype('string').astype('category')
plt.figure(figsize=(12, 6))
ax = sns.countplot(data=grd, x='IR_29301_SEVERIDAD')
plt.title('Distribución de pacientes por nivel de severidad')
plt.xlabel('Nivel de severidad')
plt.ylabel('Cantidad de pacientes')
plt.xticks(ticks=[0, 1], labels=['Moderada', 'Mayor'])

# Añadir porcentaje en las barras
total = len(grd)
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.1f}%'
    x = p.get_x() + p.get_width() / 2
    y = p.get_height() / 2
    ax.annotate(percentage, (x, y), ha='center', va='center', color='white', weight='bold')

plt.show()

In [ ]:
# Relación entre severidad y días de estancia
plt.figure(figsize=(12, 6))
sns.boxplot(data=grd, x='IR_29301_SEVERIDAD', y='DIAS_ESTANCIA')
plt.title('Días de estancia por nivel de severidad')
plt.xlabel('Nivel de severidad')
plt.ylabel('Días de estancia')
plt.xticks(ticks=[0, 1], labels=['Moderada', 'Mayor'])
plt.ylim(0, 70)  # Limitamos el eje Y para mejor visualización
plt.show()

In [ ]:
plt.hist(grd['DIAS_ESTANCIA'], bins=100)
plt.title('Distribución de días de estancia')
plt.show()

In [ ]:
grd[grd['DIAS_ESTANCIA'] >= 400]['DIAS_ESTANCIA'].value_counts()

Obtenemos ahora datos sobre la cantidad de pacientes y diagnosticos segun Servicio de Salud.

In [ ]:
# Top 10 servicios de salud con más pacientes
plt.figure(figsize=(14, 6))
top_10_servicios = grd['SERVICIO_SALUD'].value_counts().head(10)
top_10_servicios.index = top_10_servicios.index.astype('string')
sns.barplot(top_10_servicios, orient='h')
plt.title('Top 10 servicios de salud con más pacientes')
plt.ylabel('Servicio de salud')
plt.xlabel('Cantidad de pacientes')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 diagnósticos más frecuentes
plt.figure(figsize=(14, 8))
top_10_grd = grd['DIAGNOSTICO1'].value_counts().head(10)
sns.barplot(x=top_10_grd.values, y=top_10_grd.index)
plt.title('Top 10 diagnósticos GRD más frecuentes')
plt.xlabel('Cantidad de pacientes')
plt.tight_layout()
plt.show()

Se combinan variables anteriores para ver con más detalle relaciones entre severidad, edad, sexo, etc. 

In [ ]:
plt.figure(figsize=(30, 10))
crosstab = pd.crosstab(grd['SEXO'], grd['IR_29301_SEVERIDAD'])
crosstab.plot(kind='bar', stacked=True, color=sns.color_palette("pastel"))
plt.title('Distribución de niveles de severidad por sexo')
plt.xlabel('Sexo')
plt.ylabel('Cantidad de pacientes')
plt.legend(title='Nivel de severidad', fontsize=12)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Relación entre edad y severidad
plot_data = grd.dropna(subset=['IR_29301_SEVERIDAD', 'EDAD']).copy()
plot_data['IR_29301_SEVERIDAD'] = plot_data['IR_29301_SEVERIDAD'].astype('category')
plot_data['EDAD'] = plot_data['EDAD'].astype(float)

plt.figure(figsize=(12, 6))
sns.boxplot(data=plot_data, x='IR_29301_SEVERIDAD', y='EDAD')
plt.title('Distribución de edades por nivel de severidad')
plt.xlabel('Nivel de severidad')
plt.ylabel('Edad (años)')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
data = grd[grd['SEXO'].notna() & (grd['SEXO'] != 'DESCONOCIDO')].dropna(subset=['EDAD', 'IR_29301_SEVERIDAD'])
sns.boxplot(data=data, x='IR_29301_SEVERIDAD', y='EDAD', hue='SEXO')
plt.title('Distribución de edades por nivel de severidad')
plt.xlabel('Nivel de severidad')
plt.ylabel('Edad (años)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')  # Mueve la leyenda fuera del área del gráfico
plt.show()

Se comparan los top 5 hospitales según su peso promedio de GRD, que indica los costos en relación a una hospitalización promedio.

In [ ]:
# Top 5 hospitales con mayor promedio de peso GRD (mayor complejidad)
# df_hospitales = pd.read_csv("datos/Hospitales.csv", header=None, names=['COD_HOSPITAL', 'NOMBRE_HOSPITAL'], sep="|")
hospital_peso = grd.groupby('ESTABLECIMIENTO')['IR_29301_PESO'].agg(['mean', 'median', 'count'])
hospital_peso = hospital_peso[hospital_peso['count'] > 100]  # Filtramos hospitales con pocos casos
hospital_peso = hospital_peso.sort_values('mean', ascending=False).head(5)

# Calculamos la media y mediana de todos los hospitales
mean_all = grd['IR_29301_PESO'].mean()
median_all = grd['IR_29301_PESO'].median()

plt.figure(figsize=(14, 10))
ax = sns.barplot(y=hospital_peso.index, x=hospital_peso['mean'], orient='h', label='Media')
plt.title('Top 5 hospitales con mayor promedio de peso GRD')
plt.ylabel('Hospital')
plt.xlabel('Peso GRD promedio')
plt.xticks(rotation=0, ha='right')

# Añadimos líneas verticales para la media y mediana de todos los hospitales
plt.axvline(mean_all, color='r', linestyle='--', label=f'Media de todos los hospitales: {mean_all:.2f}')
plt.axvline(median_all, color='g', linestyle='-', label=f'Mediana de todos los hospitales: {median_all:.2f}')

# Añadimos el promedio dentro de la barra en color blanco
for p in ax.patches:
    width = p.get_width()
    plt.text(width - 0.1, p.get_y() + p.get_height() / 2, f'{width:.2f}', ha='center', va='center', color='white', weight='bold')

plt.legend()
plt.show()


Se busca agrupar los datos por comuna, para poder conocer los diagnósticos más frecuentes, además de filtrar por comunas que no tengan presencia de hospital GRD de alta complejidad, para identificar brechas de acceso.

In [ ]:
lejos = grd[grd['COMUNA_REGISTRADA'] != grd['COMUNA_ESTAB']]
display(lejos.groupby('COMUNA_REGISTRADA')['CIP_ENCRIPTADO'].count().sort_values(ascending=False), )

In [ ]:
lejos10 = lejos['COMUNA_REGISTRADA'].value_counts().head(10)
for comuna in lejos10.index:
    a = grd[grd['COMUNA_REGISTRADA'] == comuna]['DIAGNOSTICO1'].value_counts()
    display(a)

## Estimacion estadistica (estimadores puntuales)

En esta seccion seguimos la misma linea del EDA para obtener estimadores puntuales de parametros de interes:

- Promedios (media muestral)
- Proporciones (proporcion muestral)

Primero se calcula un resumen global del dataset cargado en `grd`.

In [ ]:
import pandas as pd

if 'grd' not in globals():
    grd = pd.read_parquet('grd_procesado.parquet')

# Parametros para promedio
vars_promedio = ['EDAD', 'DIAS_ESTANCIA', 'IR_29301_PESO']

# Parametros para proporcion 
def evento_severidad_mayor(df):
    return df['IR_29301_SEVERIDAD'].astype('string').eq('3')

def evento_sexo_femenino(df):
    return df['SEXO'].astype('string').str.upper().eq('F')

proporciones = {
    'Proporcion severidad mayor (IR_29301_SEVERIDAD = 3)': evento_severidad_mayor,
    'Proporcion sexo femenino (SEXO = F)': evento_sexo_femenino,
}

filas = []

# 1) Estimadores puntuales de medias
for var in vars_promedio:
    serie = pd.to_numeric(grd[var], errors='coerce').dropna()
    n = int(serie.shape[0])
    if n > 0:
        filas.append({
            'Parametro': f'Media poblacional de {var}',
            'Tipo_estimador': 'Media muestral',
            'Estimador_puntual': float(serie.mean()),
            'N_utilizado': n
        })

# 2) Estimadores puntuales de proporciones
for nombre_parametro, regla in proporciones.items():
    indicador = regla(grd)
    indicador = indicador.where(indicador.notna()).dropna()
    n = int(indicador.shape[0])
    if n > 0:
        filas.append({
            'Parametro': nombre_parametro,
            'Tipo_estimador': 'Proporcion muestral',
            'Estimador_puntual': float(indicador.mean()),
            'N_utilizado': n
        })

# Tabla resumen final
tabla_estimadores = pd.DataFrame(filas)
tabla_estimadores['Estimador_puntual'] = tabla_estimadores['Estimador_puntual'].round(4)

print('Tabla de resumen de estimadores puntuales:')
display(tabla_estimadores)


## Ajuste de proporciones y analisis por año

Para mantener consistencia con el EDA, primero identificamos como esta codificada la variable `SEXO` y definimos una regla para estimar la proporcion femenina.

Luego calculamos estimadores puntuales por año y agregamos intervalos de confianza al 95% para medias y proporciones.

In [ ]:
import numpy as np
import pandas as pd

if 'grd' not in globals():
    grd = pd.read_parquet('grd_procesado.parquet')

# Inspeccion rapida de codigos de sexo para definir regla coherente
sexo_vals = grd['SEXO'].astype('string').str.strip().str.upper()
print('Valores unicos observados en SEXO:')
print(sorted(sexo_vals.dropna().unique().tolist()))

# Regla: considera codigos frecuentes de femenino en fuentes administrativas
def evento_sexo_femenino_robusto(df):
    vals = df['SEXO'].astype('string').str.strip().str.upper()
    codigos_femenino = {'F', 'FEMENINO', 'MUJER', '2'}
    codigos_validos = codigos_femenino.union({'M', 'MASCULINO', 'HOMBRE', '1'})

    # Se consideran como validos solo codigos hombre/mujer esperados
    es_valido = vals.isin(codigos_validos)
    indicador = vals.isin(codigos_femenino)
    return indicador.where(es_valido)

# Recalculo de la tabla global con regla corregida para sexo
vars_promedio = ['EDAD', 'DIAS_ESTANCIA', 'IR_29301_PESO']
proporciones = {
    'Proporcion severidad mayor (IR_29301_SEVERIDAD = 3)': lambda d: d['IR_29301_SEVERIDAD'].astype('string').eq('3'),
    'Proporcion sexo femenino (regla robusta)': evento_sexo_femenino_robusto,
}

filas = []
for var in vars_promedio:
    serie = pd.to_numeric(grd[var], errors='coerce').dropna()
    n = int(serie.shape[0])
    if n > 0:
        filas.append({
            'Ambito': 'Global',
            'Parametro': f'Media poblacional de {var}',
            'Tipo_estimador': 'Media muestral',
            'Estimador_puntual': float(serie.mean()),
            'N_utilizado': n
        })

for nombre_parametro, regla in proporciones.items():
    indicador = regla(grd)
    indicador = indicador.dropna()
    n = int(indicador.shape[0])
    if n > 0:
        filas.append({
            'Ambito': 'Global',
            'Parametro': nombre_parametro,
            'Tipo_estimador': 'Proporcion muestral',
            'Estimador_puntual': float(indicador.mean()),
            'N_utilizado': n
        })

tabla_estimadores_global = pd.DataFrame(filas)
tabla_estimadores_global['Estimador_puntual'] = tabla_estimadores_global['Estimador_puntual'].round(4)

print('Tabla global actualizada:')
display(tabla_estimadores_global)

In [ ]:
# Estimadores puntuales por año
if 'Year' not in grd.columns:
    grd['Year'] = pd.to_datetime(grd['FECHA_INGRESO'], errors='coerce').dt.year

filas_year = []
for year, g in grd.dropna(subset=['Year']).groupby('Year'):
    # Medias
    for var in ['EDAD', 'DIAS_ESTANCIA', 'IR_29301_PESO']:
        serie = pd.to_numeric(g[var], errors='coerce').dropna()
        n = int(serie.shape[0])
        if n > 0:
            filas_year.append({
                'Ambito': f'Year={int(year)}',
                'Year': int(year),
                'Parametro': f'Media poblacional de {var}',
                'Tipo_estimador': 'Media muestral',
                'Estimador_puntual': float(serie.mean()),
                'N_utilizado': n
            })

    # Proporciones
    ind_sev = g['IR_29301_SEVERIDAD'].astype('string').eq('3').dropna()
    if ind_sev.shape[0] > 0:
        filas_year.append({
            'Ambito': f'Year={int(year)}',
            'Year': int(year),
            'Parametro': 'Proporcion severidad mayor (IR_29301_SEVERIDAD = 3)',
            'Tipo_estimador': 'Proporcion muestral',
            'Estimador_puntual': float(ind_sev.mean()),
            'N_utilizado': int(ind_sev.shape[0])
        })

    ind_fem = evento_sexo_femenino_robusto(g).dropna()
    if ind_fem.shape[0] > 0:
        filas_year.append({
            'Ambito': f'Year={int(year)}',
            'Year': int(year),
            'Parametro': 'Proporcion sexo femenino (regla)',
            'Tipo_estimador': 'Proporcion muestral',
            'Estimador_puntual': float(ind_fem.mean()),
            'N_utilizado': int(ind_fem.shape[0])
        })

tabla_estimadores_year = pd.DataFrame(filas_year)
tabla_estimadores_year['Estimador_puntual'] = tabla_estimadores_year['Estimador_puntual'].round(4)

print('Tabla de estimadores puntuales por anio:')
display(tabla_estimadores_year.sort_values(['Year', 'Tipo_estimador', 'Parametro']).reset_index(drop=True))

In [ ]:
# Intervalos de confianza (95%) para estimadores globales y por año

Z_95 = 1.96

def agregar_ic95(tabla):
    out = tabla.copy()
    n = pd.to_numeric(out['N_utilizado'], errors='coerce')
    est = pd.to_numeric(out['Estimador_puntual'], errors='coerce')

    # Error estandar segun tipo de estimador
    es_media = out['Tipo_estimador'].eq('Media muestral')

    se = pd.Series(np.nan, index=out.index, dtype='float64')

    for i, row in out.iterrows():
        amb = row['Ambito']
        param = row['Parametro']

        if row['Tipo_estimador'] == 'Media muestral':
            if 'EDAD' in param:
                col = 'EDAD'
            elif 'DIAS_ESTANCIA' in param:
                col = 'DIAS_ESTANCIA'
            else:
                col = 'IR_29301_PESO'

            if amb == 'Global':
                s = pd.to_numeric(grd[col], errors='coerce').dropna()
            else:
                year = int(str(amb).replace('Year=', ''))
                s = pd.to_numeric(grd.loc[grd['Year'] == year, col], errors='coerce').dropna()

            if s.shape[0] > 1:
                se.iloc[i] = s.std(ddof=1) / np.sqrt(s.shape[0])

        else:
            if pd.notna(n.iloc[i]) and n.iloc[i] > 0 and pd.notna(est.iloc[i]):
                p = est.iloc[i]
                se.iloc[i] = np.sqrt((p * (1 - p)) / n.iloc[i])

    out['IC95_inf'] = (est - Z_95 * se).round(4)
    out['IC95_sup'] = (est + Z_95 * se).round(4)

    mask_prop = out['Tipo_estimador'].eq('Proporcion muestral')
    out.loc[mask_prop, 'IC95_inf'] = out.loc[mask_prop, 'IC95_inf'].clip(lower=0)
    out.loc[mask_prop, 'IC95_sup'] = out.loc[mask_prop, 'IC95_sup'].clip(upper=1)

    return out

tabla_global_ic95 = agregar_ic95(tabla_estimadores_global)
tabla_year_ic95 = agregar_ic95(tabla_estimadores_year)

tabla_final_estimadores = pd.concat([tabla_global_ic95, tabla_year_ic95], ignore_index=True)

print('Resumen final con estimadores puntuales e IC95%:')
display(tabla_final_estimadores)

### Regresión Lineal: estudio de la desigualdad para los días de estancia hospitalaria

In [ ]:
top3grds = (grd[(grd['IR_29301_COD_GRD'].isin(['091302', '041013', '061203', '051403', '081072']))]).copy()
top3grds = top3grds[top3grds['DIAS_ESTANCIA'] != 0]
top3grds['SEXO'] = top3grds['SEXO'].astype('string')
top3grds['SEXO'] = top3grds['SEXO'].replace({'DESCONOCIDO':np.nan})
top3grds.dropna(subset=['SEXO'],inplace=True)
top3grds.reset_index(drop=True, inplace=True)
top3grds['iseow'] = (top3grds['FECHA_INGRESO'].dt.dayofweek > 4).astype(int)
top3grds = top3grds[(top3grds['TIPOALTA'] != 'FALLECIDO')]
top3grds['IR_29301_SEVERIDAD'] = top3grds['IR_29301_SEVERIDAD'].astype('string')

import statsmodels.formula.api as smf
import numpy as np

modelo = smf.ols(formula=
        "np.log(DIAS_ESTANCIA) ~ EDAD + I(EDAD**2) + C(SEXO) + IR_29301_PESO \
        + N_PROCEDIMIENTOS + N_COMORBILIDADES + IR_29301_SEVERIDAD + CRITICAS_1000HAB \
        + C(TIENE_ESTAB) + POB_MULTIDIM",
        data=top3grds)
resultado = modelo.fit()

print(resultado.summary())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Le pedimos al modelo que haga sus predicciones con los datos
predicciones_log = resultado.predict() 
valores_reales_log = top3grds['LOG_ESTANCIA'] # o como se llame en tu df

# 2. Creamos el gráfico
plt.figure(figsize=(10, 6))
sns.scatterplot(x=predicciones_log, y=valores_reales_log, alpha=0.005, color='mediumturquoise')

# 3. Dibujamos la línea roja de "predicción perfecta"
limite_min = min(predicciones_log.min(), valores_reales_log.min())
limite_max = max(predicciones_log.max(), valores_reales_log.max())
plt.plot([limite_min, limite_max], [limite_min, limite_max], color='chocolate', linewidth=2, linestyle=':')

plt.title('Desempeño del Modelo: Estancia Real vs. Predicha (Escala Logarítmica)')
plt.xlabel('Días de Estancia Predichos por el Modelo')
plt.ylabel('Días de Estancia Reales')
plt.grid(True, alpha=0.3)
plt.show()